<a href="https://colab.research.google.com/github/Khalidsyfullah/USplitVQA/blob/main/Federated_Learning/New_BioMedClip_Federated_Learning_SLAKE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "open_clip_torch", "transformers", "datasets", "openpyxl", "tqdm"])

import os, json, random, time, copy
import numpy as np
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR = "/content/results"; os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# CONFIG
# =============================================================================
class Config:
    clip_model_name = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
    vision_dim = 768; text_dim = 768; hidden_dim = 768
    num_fusion_layers = 2; num_attn_heads = 8; fusion_dropout = 0.15
    num_clients = 5; global_rounds = 20; local_epochs = 3
    batch_size = 16; learning_rate = 1e-4; weight_decay = 1e-4; label_smoothing = 0.1
    max_answer_vocab = 0; min_answer_freq = 2

cfg = Config()


# ─────────────────────────────────────────────────────────────────────
# 3. SLAKE  (mdwiratathya/SLAKE-vqa-english)
# ─────────────────────────────────────────────────────────────────────
# ~14,028 QA pairs (English subset) · 642 images
# Modalities: CT, MRI, X-Ray · Body parts: head/neck/chest/abdomen/pelvis
# Closed-ended (yes/no) + Open-ended (organ, modality, plane, position,
# abnormality, size, color, shape, KG-based questions)

def normalize_answer_slake(ans: str) -> str:
    """Normalize SLAKE answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'none', 'not sure'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Modality synonyms (SLAKE has CT/MRI/X-Ray) ──
    modality_map = {
        'ct scan': 'ct', 'ct': 'ct', 'computed tomography': 'ct',
        'cat scan': 'ct',
        'mri': 'mri', 'magnetic resonance imaging': 'mri', 'mr': 'mri',
        'mri - Loss contrast': 'mri', 't1': 'mri', 't2': 'mri',
        't1-weighted': 'mri', 't2-weighted': 'mri', 'flair': 'mri',
        'x-ray': 'x-ray', 'x ray': 'x-ray', 'xray': 'x-ray',
        'radiograph': 'x-ray', 'plain film': 'x-ray',
    }
    if ans in modality_map:
        return modality_map[ans]

    # ── Plane synonyms ──
    plane_map = {
        'axial': 'axial', 'transverse': 'axial', 'horizontal': 'axial',
        'axial plane': 'axial', 'transverse plane': 'axial',
        'coronal': 'coronal', 'frontal': 'coronal', 'coronal plane': 'coronal',
        'sagittal': 'sagittal', 'sagittal plane': 'sagittal',
    }
    if ans in plane_map:
        return plane_map[ans]

    # ── Anatomical / organ synonyms (SLAKE covers head/chest/abdomen/pelvis) ──
    anatomy_map = {
        'brain': 'brain', 'cerebral': 'brain', 'cerebrum': 'brain',
        'head': 'brain', 'cranium': 'brain',
        'lung': 'lung', 'lungs': 'lung', 'pulmonary': 'lung',
        'left lung': 'left lung', 'right lung': 'right lung',
        'heart': 'heart', 'cardiac': 'heart',
        'liver': 'liver', 'hepatic': 'liver',
        'kidney': 'kidney', 'kidneys': 'kidney', 'renal': 'kidney',
        'left kidney': 'left kidney', 'right kidney': 'right kidney',
        'spleen': 'spleen', 'splenic': 'spleen',
        'pancreas': 'pancreas', 'pancreatic': 'pancreas',
        'gallbladder': 'gallbladder', 'gall bladder': 'gallbladder',
        'stomach': 'stomach', 'gastric': 'stomach',
        'bladder': 'bladder', 'urinary bladder': 'bladder',
        'spine': 'spine', 'spinal': 'spine', 'vertebral': 'spine',
        'vertebra': 'spine', 'vertebrae': 'spine',
        'chest': 'chest', 'thorax': 'chest', 'thoracic': 'chest',
        'abdomen': 'abdomen', 'abdominal': 'abdomen',
        'pelvis': 'pelvis', 'pelvic': 'pelvis',
        'neck': 'neck', 'cervical': 'neck',
    }
    if ans in anatomy_map:
        return anatomy_map[ans]

    # ── Laterality ──
    lat_map = {
        'right side': 'right', 'right-sided': 'right',
        'left side': 'left', 'left-sided': 'left',
        'both sides': 'bilateral', 'bilateral': 'bilateral', 'both': 'bilateral',
    }
    if ans in lat_map:
        return lat_map[ans]

    # ── Abnormality synonyms ──
    abnorm_map = {
        'normal': 'normal', 'no abnormality': 'normal',
        'no abnormalities': 'normal', 'no finding': 'normal',
        'no findings': 'normal', 'unremarkable': 'normal',
        'tumor': 'tumor', 'tumour': 'tumor', 'mass': 'tumor',
        'neoplasm': 'tumor',
        'inflammation': 'inflammation', 'inflamed': 'inflammation',
        'inflammatory': 'inflammation',
        'fracture': 'fracture', 'broken': 'fracture',
        'effusion': 'effusion', 'fluid': 'effusion',
        'pleural effusion': 'pleural effusion',
        'pneumonia': 'pneumonia',
        'edema': 'edema', 'oedema': 'edema', 'swelling': 'edema',
        'hemorrhage': 'hemorrhage', 'haemorrhage': 'hemorrhage',
        'bleeding': 'hemorrhage',
        'atrophy': 'atrophy', 'atrophic': 'atrophy',
        'calcification': 'calcification', 'calcified': 'calcification',
        'enlarged': 'enlargement', 'enlargement': 'enlargement',
        'hypertrophy': 'enlargement',
    }
    if ans in abnorm_map:
        return abnorm_map[ans]

    # ── Remove articles ──
    ans = re.sub(r'^(the|a|an)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    return ans

# =============================================================================
# LOAD DATASET INTO RAM
# =============================================================================
print("\n" + "="*60 + "\nLOADING VQA-RAD INTO RAM\n" + "="*60)
from datasets import load_dataset
ds = load_dataset('mdwiratathya/SLAKE-vqa-english')
#ds = load_dataset('flaviagiammarino/vqa-rad')
#ds = load_dataset('flaviagiammarino/path-vqa')

def extract(split_data, name):
    samples = []
    for s in tqdm(split_data, desc=name):
        try:
            img = s.get('image'); q = str(s.get('question','')); a = str(s.get('answer','')).strip().lower()
            a = normalize_answer_slake(a)
            if img and q and a: samples.append({'image': img.convert('RGB'), 'question': q, 'answer': a})
        except: continue
    print(f"  {name}: {len(samples)}"); return samples

train_samples = extract(ds['train'], 'train')
test_samples = extract(ds['test'], 'test')
del ds

all_ans = [s['answer'] for s in train_samples + test_samples]
counts = Counter(all_ans)
filtered = [(a,c) for a,c in counts.most_common() if c >= cfg.min_answer_freq]
if cfg.max_answer_vocab > 0: filtered = filtered[:cfg.max_answer_vocab]
answer_vocab = {a: i for i, (a,_) in enumerate(sorted(filtered, key=lambda x: x[0]))}
if '<unk>' not in answer_vocab: answer_vocab['<unk>'] = len(answer_vocab)
num_classes = len(answer_vocab)
print(f"  Vocab: {num_classes}, Train: {len(train_samples)}, Test: {len(test_samples)}")

# =============================================================================
# LOAD BiomedCLIP
# =============================================================================
print("\n" + "="*60 + "\nLOADING BiomedCLIP\n" + "="*60)
from open_clip import create_model_and_transforms, get_tokenizer
clip_model, _, preprocess_val = create_model_and_transforms(cfg.clip_model_name)
tokenizer = get_tokenizer(cfg.clip_model_name)
clip_model = clip_model.to(device)

# =============================================================================
# DATASET
# =============================================================================
class VQADataset(Dataset):
    def __init__(self, samples, vocab, preprocess, tokenizer):
        self.samples=samples; self.vocab=vocab; self.preprocess=preprocess; self.tokenizer=tokenizer
        self.unk=vocab.get('<unk>',0)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return (self.preprocess(s['image']), self.tokenizer([s['question']])[0], self.vocab.get(s['answer'], self.unk))

def collate_fn(batch):
    imgs, txts, lbls = zip(*batch); imgs = torch.stack(imgs)
    mx = max(t.shape[0] for t in txts)
    padded = torch.zeros(len(txts), mx, dtype=txts[0].dtype)
    for i, t in enumerate(txts): padded[i, :t.shape[0]] = t
    return imgs, padded, torch.tensor(lbls, dtype=torch.long)

# Distribute IID
idx = np.random.permutation(len(train_samples)); sz = len(train_samples) // cfg.num_clients
client_splits = {c: idx[c*sz: (c+1)*sz if c < cfg.num_clients-1 else len(train_samples)].tolist() for c in range(cfg.num_clients)}

client_loaders, client_sizes = {}, {}
for cid, indices in client_splits.items():
    ds = VQADataset([train_samples[i] for i in indices], answer_vocab, preprocess_val, tokenizer)
    client_loaders[cid] = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
    client_sizes[cid] = len(indices)
    print(f"  Client {cid}: {len(indices)} samples")

test_loader = DataLoader(VQADataset(test_samples, answer_vocab, preprocess_val, tokenizer),
                          batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# =============================================================================
# MODEL — identical to centralized
# =============================================================================
class ClientEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__(); self.clip_model = clip_model
        for p in self.clip_model.parameters(): p.requires_grad = False
    @torch.no_grad()
    def encode(self, images, input_ids):
        ve = self.clip_model.visual
        x = ve.trunk.patch_embed(images); x = ve.trunk._pos_embed(x); x = ve.trunk.patch_drop(x)
        x = ve.trunk.norm_pre(x); x = ve.trunk.blocks(x); vis = ve.trunk.norm(x)
        te = self.clip_model.text; amask = (input_ids != 0).long()
        return vis, te.transformer(input_ids=input_ids, attention_mask=amask).last_hidden_state, amask

class FusionTransformerLayer(nn.Module):
    def __init__(self, dim, n_heads, dropout=0.1):
        super().__init__()
        self.v2t_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.v2t_norm1 = nn.LayerNorm(dim)
        self.v2t_ffn = nn.Sequential(nn.Linear(dim,dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4,dim), nn.Dropout(dropout))
        self.v2t_norm2 = nn.LayerNorm(dim)
        self.t2v_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.t2v_norm1 = nn.LayerNorm(dim)
        self.t2v_ffn = nn.Sequential(nn.Linear(dim,dim*4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim*4,dim), nn.Dropout(dropout))
        self.t2v_norm2 = nn.LayerNorm(dim)
    def forward(self, v, t, text_key_padding_mask=None):
        o, _ = self.v2t_attn(v, t, t, key_padding_mask=text_key_padding_mask)
        v = self.v2t_norm1(v+o); v = self.v2t_norm2(v + self.v2t_ffn(v))
        o, _ = self.t2v_attn(t, v, v); t = self.t2v_norm1(t+o); t = self.t2v_norm2(t + self.t2v_ffn(t))
        return v, t

class ServerModel(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__(); D = cfg.hidden_dim
        self.vis_proj = nn.Linear(cfg.vision_dim, D) if cfg.vision_dim != D else nn.Identity()
        self.txt_proj = nn.Linear(cfg.text_dim, D) if cfg.text_dim != D else nn.Identity()
        self.fusion_layers = nn.ModuleList([FusionTransformerLayer(D, cfg.num_attn_heads, cfg.fusion_dropout) for _ in range(cfg.num_fusion_layers)])
        self.pool_query = nn.Parameter(torch.randn(1,1,D)*0.02)
        self.pool_attn = nn.MultiheadAttention(D, cfg.num_attn_heads, dropout=cfg.fusion_dropout, batch_first=True)
        self.pool_norm = nn.LayerNorm(D)
        self.head = nn.Sequential(nn.Linear(D,D), nn.GELU(), nn.Dropout(cfg.fusion_dropout),
                                   nn.Linear(D,D//2), nn.GELU(), nn.Dropout(cfg.fusion_dropout), nn.Linear(D//2, num_classes))
    def forward(self, vis, txt, text_mask=None):
        v = self.vis_proj(vis); t = self.txt_proj(txt)
        kpm = (text_mask == 0) if text_mask is not None else None
        for fl in self.fusion_layers: v, t = fl(v, t, text_key_padding_mask=kpm)
        combined = torch.cat([v, t], dim=1); B = combined.shape[0]
        pq = self.pool_query.expand(B,-1,-1)
        pooled, _ = self.pool_attn(pq, combined, combined)
        return self.head(self.pool_norm(pq + pooled).squeeze(1))

client_enc = ClientEncoder(clip_model).to(device)
global_model = ServerModel(cfg, num_classes).to(device)
n_params = sum(p.numel() for p in global_model.parameters())
print(f"\n  Server model: {n_params:,} ({n_params/1e6:.1f}M)")

# =============================================================================
# FedAvg TRAINING
# =============================================================================
print("\n" + "="*60 + "\nTRAINING (FedAvg)\n" + "="*60)

def get_params(m): return [p.data.cpu().numpy().copy() for p in m.parameters()]
def set_params(m, params):
    for p, w in zip(m.parameters(), params): p.data = torch.from_numpy(w).to(p.device)
def fedavg_agg(cp, sizes):
    total = sum(sizes); wts = [n/total for n in sizes]
    return [sum(wts[i]*cp[i][p] for i in range(len(cp))) for p in range(len(cp[0]))]

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)

@torch.no_grad()
def eval_model(loader):
    global_model.eval(); loss_sum, correct, total = 0.0, 0, 0
    for imgs, txts, lbls in loader:
        imgs, txts, lbls = imgs.to(device), txts.to(device), lbls.to(device)
        vis, txt, amask = client_enc.encode(imgs, txts)
        logits = global_model(vis, txt, text_mask=amask)
        loss_sum += criterion(logits, lbls).item()*lbls.size(0)
        correct += (logits.argmax(-1)==lbls).sum().item(); total += lbls.size(0)
    return loss_sum/total, 100*correct/total

history = {'round':[], 'avg_train_loss':[], 'avg_train_acc':[], 'test_loss':[], 'test_acc':[], 'round_time':[]}
# ── ADDED TIME HISTORY INITIALIZATION ──
for cid in range(cfg.num_clients):
    history[f'client_{cid}_loss'] = []; history[f'client_{cid}_acc'] = []; history[f'client_{cid}_time'] = []

best_acc, best_state = 0.0, None

for rnd in range(1, cfg.global_rounds + 1):
    t0 = time.time(); gp = get_params(global_model)
    round_cp, round_losses, round_correct, round_total = [], [], 0, 0

    pbar = tqdm(range(cfg.num_clients), desc=f"Round {rnd:2d}/{cfg.global_rounds}", leave=False)
    for cid in pbar:
        c_t0 = time.time() # ── START CLIENT TIMER ──

        local = copy.deepcopy(global_model); set_params(local, [p.copy() for p in gp]); local.train()
        opt = torch.optim.AdamW(local.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        c_loss, c_correct, c_total = 0.0, 0, 0
        for _ in range(cfg.local_epochs):
            for imgs, txts, lbls in client_loaders[cid]:
                imgs, txts, lbls = imgs.to(device), txts.to(device), lbls.to(device)
                vis, txt, amask = client_enc.encode(imgs, txts)
                opt.zero_grad(); logits = local(vis, txt, text_mask=amask)
                loss = criterion(logits, lbls); loss.backward()
                nn.utils.clip_grad_norm_(local.parameters(), 1.0); opt.step()
                c_loss += loss.item()*lbls.size(0)
                c_correct += (logits.argmax(-1)==lbls).sum().item(); c_total += lbls.size(0)

        c_time = time.time() - c_t0 # ── END CLIENT TIMER ──

        round_cp.append(get_params(local))
        cl = c_loss/max(c_total,1); ca = 100*c_correct/max(c_total,1)
        round_losses.append(cl); round_correct += c_correct; round_total += c_total

        history[f'client_{cid}_loss'].append(round(cl, 4))
        history[f'client_{cid}_acc'].append(round(ca, 2))
        history[f'client_{cid}_time'].append(round(c_time, 2)) # ── SAVE TO HISTORY ──

        # ── ADDED TIME TO TQDM POSTFIX ──
        pbar.set_postfix(client=cid, loss=f"{cl:.4f}", acc=f"{ca:.1f}%", t=f"{c_time:.1f}s")
        del local, opt

    set_params(global_model, fedavg_agg(round_cp, list(client_sizes.values())))
    te_l, te_a = eval_model(test_loader)
    rt = time.time() - t0
    avg_l = np.mean(round_losses); avg_a = 100*round_correct/max(round_total,1)

    history['round'].append(rnd); history['avg_train_loss'].append(round(avg_l, 4))
    history['avg_train_acc'].append(round(avg_a, 2)); history['test_loss'].append(round(te_l, 4))
    history['test_acc'].append(round(te_a, 2)); history['round_time'].append(round(rt, 1))

    marker = ""
    if te_a > best_acc: best_acc = te_a; best_state = copy.deepcopy(global_model.state_dict()); marker = " ★"
    print(f"Round {rnd:2d}/{cfg.global_rounds} [{rt:.1f}s]  Train: {avg_l:.4f}/{avg_a:.1f}%  Test: {te_l:.4f}/{te_a:.1f}%{marker}")

if best_state: global_model.load_state_dict(best_state)
te_l, te_a = eval_model(test_loader)
print(f"\n{'='*60}\nFINAL: {te_a:.2f}% (best: {best_acc:.2f}%)\n{'='*60}")

# =============================================================================
# SAVE TO EXCEL
# =============================================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

wb = openpyxl.Workbook(); ws = wb.active; ws.title = "FedAvg VQA-RAD"
hfont = Font(name='Arial', bold=True, size=11, color='FFFFFF')
hfill = PatternFill(start_color='1A5276', end_color='1A5276', fill_type='solid')

# ── ADDED TIME TO HEADERS ──
headers = ['Round', 'Avg Train Loss', 'Avg Train Acc (%)', 'Test Loss', 'Test Acc (%)', 'Time (s)']
for cid in range(cfg.num_clients):
    headers += [f'C{cid} Loss', f'C{cid} Acc (%)', f'C{cid} Time (s)']

for col, h in enumerate(headers, 1):
    c = ws.cell(row=1, column=col, value=h); c.font = hfont; c.fill = hfill; c.alignment = Alignment(horizontal='center')

for i, rnd in enumerate(history['round']):
    row = i + 2
    ws.cell(row=row, column=1, value=rnd)
    ws.cell(row=row, column=2, value=history['avg_train_loss'][i])
    ws.cell(row=row, column=3, value=history['avg_train_acc'][i])
    ws.cell(row=row, column=4, value=history['test_loss'][i])
    ws.cell(row=row, column=5, value=history['test_acc'][i])
    ws.cell(row=row, column=6, value=history['round_time'][i])

    # ── MODIFIED COLUMN OFFSETS TO INCLUDE TIME (3 columns per client) ──
    for cid in range(cfg.num_clients):
        ws.cell(row=row, column=7+cid*3, value=history[f'client_{cid}_loss'][i])
        ws.cell(row=row, column=8+cid*3, value=history[f'client_{cid}_acc'][i])
        ws.cell(row=row, column=9+cid*3, value=history[f'client_{cid}_time'][i])

ws2 = wb.create_sheet("Summary")
for i, (k, v) in enumerate([("Method","FedAvg"),("Dataset","VQA-RAD"),("Clients",cfg.num_clients),
    ("Local Epochs",cfg.local_epochs),("Rounds",cfg.global_rounds),("Best Test Acc",round(best_acc,2)),
    ("Final Test Acc",round(te_a,2)),("Server Params",f"{n_params:,}"),("Batch Size",cfg.batch_size)], 1):
    ws2.cell(row=i, column=1, value=k).font = Font(bold=True, name='Arial')
    ws2.cell(row=i, column=2, value=v)

for s in [ws, ws2]:
    for col in s.columns:
        s.column_dimensions[col[0].column_letter].width = max(len(str(c.value or '')) for c in col) + 2

xlsx_path = f"{OUTPUT_DIR}/slake_federated_results.xlsx"
wb.save(xlsx_path); print(f"\nResults saved to {xlsx_path}\nDONE!")